## 0. 환경설정 - 드레스 챗봇

In [ ]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python gradio python-dotenv -q

In [ ]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드

from neo4j import GraphDatabase, basic_auth
from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
from openai import OpenAI
import gradio as gr

print("모듈 로드 완료")

In [ ]:
# OPENAI_API_KEY는 .env에서 자동 로드됨
client = OpenAI()

## 1. 데이터 로드 - 드레스

In [ ]:
_BASE = os.path.abspath(os.getcwd())
DATA_FILE = os.path.join(_BASE, "json", "iwedding_dress_detail.json")

with open(DATA_FILE, "r", encoding="utf-8") as f:
    raw = json.load(f)

# JSON 구조에 따라 리스트 추출
if isinstance(raw, list):
    all_data = raw
else:
    for key in ["data", "items", "result", "partners"]:
        if key in raw and isinstance(raw[key], list):
            all_data = raw[key]
            break
    else:
        all_data = list(raw.values())[0] if raw else []

print(f"드레스: {len(all_data)}개 로드")
prices = [d.get("salePrice",0) for d in all_data if d.get("salePrice",0) > 0]
if prices:
    print(f"가격 범위: {min(prices):,}원 ~ {max(prices):,}원 (평균 {sum(prices)//len(prices):,}원)")
print(f"업체 수: {len(set(d.get('name','') for d in all_data))}개")

## 2. Neo4j 연결 + 데이터 삽입

In [ ]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "password123")

driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    print(f"Neo4j 현재 노드 수: {cnt}")

In [ ]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("기존 데이터 삭제 완료")
    for c in [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (v:Vendor) REQUIRE v.partnerId IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (r:Region) REQUIRE r.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Tag) REQUIRE t.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (sf:StyleFilter) REQUIRE sf.name IS UNIQUE",
    ]:
        session.run(c)
    print("제약조건 생성 완료")

In [ ]:
def insert_batch(session, items, batch_size=100):
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        params = []
        for item in batch:
            params.append({
                "partnerId": item.get("partnerId", 0),
                "uuid": item.get("uuid", ""),
                "name": item.get("name", ""),
                "typeName": item.get("typeName", ""),
                "tel": item.get("tel", ""),
                "address": item.get("address", ""),
                "region": item.get("region", "") or "기타",
                "coverUrl": item.get("coverUrl", ""),
                "profile": item.get("profile", ""),
                "profileUrl": item.get("profileUrl", ""),
                "rating": item.get("rating", 0),
                "reviewCnt": item.get("reviewCnt", 0),
                "productPrice": item.get("productPrice", 0),
                "salePrice": item.get("salePrice", 0),
                "eventOptionPrice": item.get("eventOptionPrice", 0),
                "eventPrice": item.get("eventPrice", 0),
                "likeCnt": item.get("likeCnt", 0),
                "viewCnt": item.get("viewCnt", 0),
                "orderCnt": item.get("orderCnt", 0),
                "holiday": item.get("holiday", ""),
                "packageInfoStr": json.dumps(item.get("packageInfo", []), ensure_ascii=False),
                "addcostStr": json.dumps(item.get("addcostOptions", []), ensure_ascii=False),
                "reviewsStr": json.dumps(item.get("reviews", []), ensure_ascii=False),
                "detailCmt": item.get("detailCmt", ""),
                "iweddingNo": item.get("iwedding_no", ""),
                "enterpriseCode": item.get("iwedding_enterprise_code", ""),
                "productName": item.get("iwedding_product_name", ""),
                "subCategory": item.get("iwedding_sub_category", ""),
                "tags": [t["name"] for t in item.get("tags", []) if t.get("name")],
                "styles": [sf["name"] for sf in item.get("styleFilter", []) if sf.get("name")],
            })

        session.run("""
            UNWIND $batch AS row
            MERGE (v:Vendor {partnerId: row.partnerId})
            SET v += {uuid: row.uuid, name: row.name, typeName: row.typeName,
                tel: row.tel, address: row.address, region: row.region,
                coverUrl: row.coverUrl, profile: row.profile, profileUrl: row.profileUrl,
                rating: row.rating, reviewCnt: row.reviewCnt,
                productPrice: row.productPrice, salePrice: row.salePrice,
                eventOptionPrice: row.eventOptionPrice, eventPrice: row.eventPrice,
                likeCnt: row.likeCnt, viewCnt: row.viewCnt, orderCnt: row.orderCnt,
                holiday: row.holiday, packageInfoStr: row.packageInfoStr,
                addcostStr: row.addcostStr, reviewsStr: row.reviewsStr,
                detailCmt: row.detailCmt, iweddingNo: row.iweddingNo,
                enterpriseCode: row.enterpriseCode, productName: row.productName,
                subCategory: row.subCategory}
        """, batch=params)

        session.run("""
            UNWIND $batch AS row
            MATCH (v:Vendor {partnerId: row.partnerId})
            MERGE (r:Region {name: row.region})
            MERGE (v)-[:IN_REGION]->(r)
        """, batch=params)

        session.run("""
            UNWIND $batch AS row
            MATCH (v:Vendor {partnerId: row.partnerId})
            UNWIND row.tags AS tagName
            MERGE (t:Tag {name: tagName})
            MERGE (v)-[:HAS_TAG]->(t)
        """, batch=params)

        session.run("""
            UNWIND $batch AS row
            MATCH (v:Vendor {partnerId: row.partnerId})
            UNWIND row.styles AS styleName
            MERGE (sf:StyleFilter {name: styleName})
            MERGE (v)-[:HAS_STYLE]->(sf)
        """, batch=params)

        print(f"  {min(i+batch_size, len(items))}/{len(items)}")

with driver.session() as session:
    insert_batch(session, all_data)
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    rel = session.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]
    print(f"\n삽입 완료 - 노드: {cnt}개, 관계: {rel}개")


## 3. MySQL (더미 fallback)

In [ ]:
import mysql.connector

MYSQL_HOST = os.environ.get("MYSQL_HOST", "")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
COUPLE_ID = 2

conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST, user=MYSQL_USER,
            password=MYSQL_PASSWORD, database=MYSQL_DB,
            port=int(os.environ.get("MYSQL_PORT", 3306))
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 - 더미 사용")

def get_user_preference(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_preferences WHERE couple_id = %s", (couple_id,))
            row = cur.fetchone(); cur.close()
            return row
        except: pass
    return {'couple_id': 2, 'region': '서울', 'sub_region': '강남구', 'style': '클래식', 'budget': '190만원', 'category': '드레스'}

def get_user_likes(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_venue_likes WHERE couple_id = %s", (couple_id,))
            rows = cur.fetchall(); cur.close()
            return rows
        except: pass
    return [{"partner_id": all_data[0]["partnerId"], "name": all_data[0]["name"], "category": "드레스"}]

print("사용자 함수 정의 완료")

## 4. Neo4j 스키마 추출

In [ ]:
def get_schema(uri, user, password):
    d = GraphDatabase.driver(uri, auth=basic_auth(user, password))
    with d.session() as session:
        nodes = session.run("MATCH (n) WITH DISTINCT labels(n) AS lbls, keys(n) AS ks, n UNWIND lbls AS l UNWIND ks AS k RETURN l, k, n[k] AS sv")
        rels = session.run("MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS sl, type(r) AS rt, labels(b) AS el")
        schema = {"nodes": {}, "relations": []}
        for rec in nodes:
            l, k = rec["l"], rec["k"]
            if l not in schema["nodes"]: schema["nodes"][l] = {}
            v = rec["sv"]
            t = "STRING" if isinstance(v, str) else "INTEGER" if isinstance(v, int) else "FLOAT" if isinstance(v, float) else "UNKNOWN"
            schema["nodes"][l][k] = t
        for rec in rels:
            sl = rec["sl"][0] if rec["sl"] else "?"
            el = rec["el"][0] if rec["el"] else "?"
            schema["relations"].append(f"(:{sl})-[:{rec['rt']}]->(:{el})")
    d.close()
    return schema

def format_schema(schema):
    lines = ["Node properties:"]
    for label, props in schema["nodes"].items():
        ps = ", ".join(f"{k}: {v}" for k, v in props.items())
        lines.append(f"  {label} {{{ps}}}")
    lines.append("Relationships:")
    for r in schema["relations"]: lines.append(f"  {r}")
    return "\n".join(lines)

schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)
print(neo4j_schema)


## 5. LLM Intent 파서 - 드레스 (30개 학습 데이터)

In [ ]:
PARSE_SYSTEM_PROMPT = """당신은 웨딩 드레스 추천 챗봇의 입력 분석기입니다.
사용자 메시지를 분석하여 반드시 아래 JSON 형식으로만 출력하세요. 다른 텍스트 없이 JSON만.

{
  "intent": "RECOMMEND | COMPARE | INFO | PREFERENCE_QUERY | LIKE_QUERY | LIKE_BASED_RECOMMEND | GENERAL",
  "conditions": {
    "maxPrice": null,
    "minPrice": null,
    "region": null,
    "styles": [],
    "exclude": [],
    "mustHave": []
  },
  "priority": [],
  "comparison": [],
  "refersPrevious": false,
  "sentiment": "exploring | decided"
}

=== Intent 분류 규칙 ===

[RECOMMEND] 추천 요청
- "~추천해줘", "~있을까", "~찾아줘", "~알려줘"
- 가격대를 나눠서 "각각" 추천하는 것도 RECOMMEND (COMPARE 아님!)
- "~이하랑 ~이상 각각 추천해줘"는 RECOMMEND (comparison에 넣지 마세요)

[COMPARE] 특정 업체명 2개 이상을 직접 비교할 때만
- "셀렉션H랑 모네뜨아르 비교해줘" -> COMPARE
- 반드시 업체 고유명이 2개 이상 있어야 COMPARE

[INFO] 특정 업체 1개의 상세 정보 질문
- "셀렉션H 가격 알려줘" -> INFO, comparison: ["셀렉션H"]

[PREFERENCE_QUERY] "내 취향", "내 스타일"
[LIKE_QUERY] "좋아요한", "찜한"
[LIKE_BASED_RECOMMEND] "찜한 것과 비슷한 곳"
[GENERAL] 웨딩 일반 질문, 인사

=== 데이터 범위 ===
현재 보유 데이터는 서울/경기 지역 드레스샵 88개 업체(268개 상품)입니다.
부산, 대구, 대전 등 지방 데이터는 없습니다.

=== 가격 해석 ===
- 드레스 평균: 약 190만원
- "저렴한" -> maxPrice: 1300000
- "너무 비싸지 않은" -> maxPrice: 1900000
- "고급", "프리미엄" -> minPrice: 2500000
- "100만원 이하" -> maxPrice: 1000000
- "100~200만원" -> minPrice: 1000000, maxPrice: 2000000
- "100만원대" -> minPrice: 1000000, maxPrice: 1999999
- "200만원대" -> minPrice: 2000000, maxPrice: 2999999

=== 스타일/조건 키워드 ===
촬영드레스, 본식드레스, 촬영+본식, 화이트, 컬러, 미니드레스,
A라인, 머메이드, 프린세스, 클래식, 러블리, 모던, 빈티지, 심플,
2벌, 3벌, 4벌, 5벌, 프리미엄, 수입드레스

=== 우선순위 ===
- "가격이 제일 중요" -> priority: ["price"]
- "디자인 위주로" -> priority: ["style"]
- "리뷰 좋은 곳" -> priority: ["rating"]
- "인기있는" -> priority: ["popular"]

=== 제외 조건 ===
"~말고", "~싫어", "~빼고" -> exclude

=== 이전 대화 참조 ===
"그 중에서", "아까 추천한", "거기" -> refersPrevious: true

=== 학습 예시 (30개) ===
입력: "드레스 추천해줘" -> {"intent":"RECOMMEND","conditions":{},"priority":[]}
입력: "200만원 이하 드레스 추천" -> {"intent":"RECOMMEND","conditions":{"maxPrice":2000000},"priority":["price"]}
입력: "촬영+본식 드레스 4벌 추천" -> {"intent":"RECOMMEND","conditions":{"styles":["촬영+본식","4벌"]}}
입력: "본식 드레스만 따로 있는 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["본식"]}}
입력: "리뷰 좋은 드레스샵" -> {"intent":"RECOMMEND","priority":["rating"]}
입력: "150만원 이하랑 이상 각각 추천해줘" -> {"intent":"RECOMMEND","conditions":{"maxPrice":1500000},"priority":["price"]}
입력: "셀렉션H랑 모네뜨아르 비교해줘" -> {"intent":"COMPARE","comparison":["셀렉션H","모네뜨아르"]}
입력: "셀렉션H 가격 알려줘" -> {"intent":"INFO","comparison":["셀렉션H"]}
입력: "내 취향 알려줘" -> {"intent":"PREFERENCE_QUERY"}
입력: "좋아요한 업체" -> {"intent":"LIKE_QUERY"}
입력: "그 중에서 리뷰 좋은 곳은?" -> {"intent":"RECOMMEND","refersPrevious":true,"priority":["rating"]}
입력: "부산 드레스샵 추천" -> {"intent":"RECOMMEND","conditions":{"region":"부산"}}
입력: "100만원대 가성비 좋은 곳" -> {"intent":"RECOMMEND","conditions":{"minPrice":1000000,"maxPrice":1999999},"priority":["price"]}
입력: "요즘 인기있는 곳" -> {"intent":"RECOMMEND","priority":["popular"]}
입력: "화이트 드레스 많은 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["화이트"]}}
입력: "컬러 드레스 있는 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["컬러"]}}
입력: "수입 드레스 취급하는 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["수입"]}}
입력: "촬영 드레스 3벌 이상" -> {"intent":"RECOMMEND","conditions":{"styles":["촬영","3벌"]}}
입력: "프리미엄 드레스샵" -> {"intent":"RECOMMEND","conditions":{"minPrice":2500000}}
입력: "할인 많이 된 곳" -> {"intent":"RECOMMEND","priority":["price"]}
입력: "가장 저렴한 드레스 5개" -> {"intent":"RECOMMEND","priority":["price"]}
입력: "드레스 보통 얼마야?" -> {"intent":"GENERAL"}
입력: "찜한 것과 비슷한 곳" -> {"intent":"LIKE_BASED_RECOMMEND"}
입력: "본식 드레스 화이트만" -> {"intent":"RECOMMEND","conditions":{"styles":["본식","화이트"]}}
입력: "A라인 드레스 있는 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["A라인"]}}
입력: "머메이드 라인 추천" -> {"intent":"RECOMMEND","conditions":{"styles":["머메이드"]}}
입력: "누벨드블랑 패키지 구성은?" -> {"intent":"INFO","comparison":["누벨드블랑"]}
입력: "조슈아벨이랑 크리드제이 뭐가 나아?" -> {"intent":"COMPARE","comparison":["조슈아벨","크리드제이"]}
입력: "드레스 벌수 많은 곳" -> {"intent":"RECOMMEND","conditions":{"styles":["5벌"]},"priority":["style"]}
입력: "사진 잘 나오는 드레스 추천" -> {"intent":"RECOMMEND","priority":["rating"]}
"""

def parse_intent(message, chat_history=None):
    messages = [{"role": "system", "content": PARSE_SYSTEM_PROMPT}]
    if chat_history:
        for h in chat_history[-4:]:
            if isinstance(h, dict):
                messages.append({"role": h.get("role","user"), "content": str(h.get("content",""))[:500]})
    messages.append({"role": "user", "content": message})
    try:
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0, max_tokens=500)
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        return json.loads(raw)
    except Exception as e:
        print(f"파싱 실패: {e}")
        return {"intent": "GENERAL", "conditions": {}, "priority": [], "comparison": [], "refersPrevious": False, "sentiment": "exploring"}

# 테스트
for msg in ["드레스 추천해줘", "200만원 이하 촬영+본식", "셀렉션H 가격 알려줘", "부산 드레스샵", "그 중에서 싼 곳"]:
    r = parse_intent(msg)
    print(f"입력: {msg}")
    print(f"  intent={r.get('intent')}, cond={r.get('conditions',{})}")


## 6. Few-shot Cypher 예시 - 드레스 (30개)

In [ ]:
examples = [
    # === 기본 추천 ===
    """USER INPUT: '드레스 추천해줘'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '웨딩드레스 알아보고 있어'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '괜찮은 드레스샵 있을까?'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0 AND v.rating >= 90
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '평점 좋은 드레스샵 추천'
QUERY:
MATCH (v:Vendor) WHERE v.rating > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '리뷰 많은 드레스샵 추천해줘'
QUERY:
MATCH (v:Vendor) WHERE v.reviewCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.reviewCnt DESC, v.rating DESC LIMIT 10""",

    # === 가격 기반 ===
    """USER INPUT: '200만원 이하 드레스 추천해줘'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '100만원에서 150만원 사이 드레스'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice >= 1000000 AND v.salePrice <= 1500000
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '100만원대 가성비 좋은 드레스'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice >= 1000000 AND v.salePrice < 2000000
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.salePrice ASC LIMIT 10""",

    """USER INPUT: '가장 저렴한 드레스 5개'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    """USER INPUT: '고급 프리미엄 드레스'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice >= 2500000
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '할인 많이 된 드레스'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0 AND v.productPrice > v.salePrice
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, (v.productPrice - v.salePrice) AS discount, v.rating AS rating, v.profileUrl AS url
ORDER BY discount DESC LIMIT 10""",

    # === 스타일/구성 기반 ===
    """USER INPUT: '촬영+본식 드레스 4벌 이상 추천'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '4벌' OR t.name CONTAINS '5벌' OR v.detailCmt CONTAINS '4벌' OR v.detailCmt CONTAINS '5벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '본식 드레스만 따로 있는 곳'
QUERY:
MATCH (v:Vendor)
WHERE v.subCategory CONTAINS '본식' OR v.productName CONTAINS '본식'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영 드레스 3벌 구성'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '3벌' OR v.detailCmt CONTAINS '3벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '컬러 드레스 있는 곳'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '컬러' OR v.detailCmt CONTAINS '컬러'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '수입 드레스 취급하는 곳'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '수입' OR t.name CONTAINS '프리미엄' OR v.detailCmt CONTAINS '수입'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    # === 복합 조건 ===
    """USER INPUT: '200만원 이하 촬영+본식 드레스 4벌'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE v.salePrice <= 2000000 AND v.salePrice > 0 AND (t.name CONTAINS '4벌' OR v.detailCmt CONTAINS '4벌')
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '강남에서 150만원 이하 드레스'
QUERY:
MATCH (v:Vendor)-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남' AND v.salePrice <= 1500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    # === 제외 조건 ===
    """USER INPUT: '본식 말고 촬영 드레스만'
QUERY:
MATCH (v:Vendor)
WHERE v.productName CONTAINS '촬영' AND NOT v.productName CONTAINS '본식' AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '화이트만 말고 컬러도 포함된 곳'
QUERY:
MATCH (v:Vendor) WHERE v.detailCmt CONTAINS '컬러' AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    # === 업체 비교 ===
    """USER INPUT: '셀렉션H랑 모네뜨아르 비교해줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '셀렉션' OR v.name CONTAINS '모네뜨'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.detailCmt AS description, v.packageInfoStr AS packageInfo, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '조슈아벨이랑 크리드제이 뭐가 나아?'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '조슈아' OR v.name CONTAINS '크리드'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.detailCmt AS description, v.packageInfoStr AS packageInfo, collect(DISTINCT t.name) AS tags""",

    # === 업체 상세 ===
    """USER INPUT: '셀렉션H 가격 알려줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '셀렉션'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.name AS name, v.productPrice AS originalPrice, v.salePrice AS salePrice, v.eventPrice AS eventPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.tel AS tel, v.detailCmt AS description, v.packageInfoStr AS packageInfo, v.addcostStr AS addcost, v.reviewsStr AS reviews, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '누벨드블랑 패키지 구성은?'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '누벨'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.name AS name, v.productPrice AS originalPrice, v.salePrice AS salePrice, v.rating AS rating, v.detailCmt AS description, v.packageInfoStr AS packageInfo, v.addcostStr AS addcost, collect(DISTINCT t.name) AS tags""",

    # === 인기/트렌드 ===
    """USER INPUT: '요즘 인기있는 드레스샵'
QUERY:
MATCH (v:Vendor) WHERE v.orderCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.orderCnt AS orders, v.likeCnt AS likes, v.profileUrl AS url
ORDER BY v.orderCnt DESC, v.likeCnt DESC LIMIT 10""",

    """USER INPUT: '찜 많이 받은 드레스샵'
QUERY:
MATCH (v:Vendor) WHERE v.likeCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.likeCnt AS likes, v.profileUrl AS url
ORDER BY v.likeCnt DESC LIMIT 10""",

    # === 추가 ===
    """USER INPUT: '드레스 벌수 많은 곳'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '5벌' OR t.name CONTAINS '6벌' OR v.detailCmt CONTAINS '5벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '너무 비싸지 않은 적당한 드레스'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice <= 1900000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영+본식 세트로 되는 곳'
QUERY:
MATCH (v:Vendor) WHERE v.productName CONTAINS '촬영' AND v.productName CONTAINS '본식'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '화이트 드레스 3벌 이상'
QUERY:
MATCH (v:Vendor) WHERE v.detailCmt CONTAINS '화이트' AND (v.detailCmt CONTAINS '3벌' OR v.detailCmt CONTAINS '4벌')
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",
]

print(f"Few-shot 예시 {len(examples)}개 로드 완료")


## 7. LLM / Retriever / GraphRAG

In [ ]:
llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0, "max_tokens": 2000})
retriever = Text2CypherRetriever(driver=driver, llm=llm, neo4j_schema=neo4j_schema, examples=examples)
rag = GraphRAG(retriever=retriever, llm=llm)
print("GraphRAG 초기화 완료")

## 8. 검색 + Fallback 함수

In [ ]:
AVAILABLE_REGIONS = ["서울", "강남", "서초", "송파", "성동", "용산", "마포", "광진", "중구",
                     "서대문", "동대문", "중랑", "성북", "경기", "하남", "광주", "남양주"]

def search_by_conditions(driver, parsed):
    conds = parsed.get("conditions", {})
    max_price = conds.get("maxPrice")
    min_price = conds.get("minPrice")
    styles = conds.get("styles", [])
    excludes = conds.get("exclude", [])
    region = conds.get("region")
    priority = parsed.get("priority", [])

    where_parts = ["v.salePrice > 0"]
    params = {}

    if max_price:
        where_parts.append("v.salePrice <= $maxPrice")
        params["maxPrice"] = max_price
    if min_price:
        where_parts.append("v.salePrice >= $minPrice")
        params["minPrice"] = min_price

    match_extra = ""
    if region:
        match_extra += "\nMATCH (v)-[:IN_REGION]->(r:Region)"
        where_parts.append("r.name CONTAINS $region")
        params["region"] = region

    style_clause = ""
    if styles:
        match_extra += "\nOPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)"
        match_extra += "\nOPTIONAL MATCH (v)-[:HAS_STYLE]->(sf:StyleFilter)"
        sc = []
        for i, s in enumerate(styles):
            pk = f"style{i}"
            sc.append(f"t.name CONTAINS ${pk} OR sf.name CONTAINS ${pk}")
            params[pk] = s
        style_clause = "AND (" + " OR ".join(sc) + ")"

    order = "v.rating DESC, v.reviewCnt DESC"
    if "price" in priority: order = "v.salePrice ASC, v.rating DESC"
    elif "popular" in priority: order = "v.orderCnt DESC, v.likeCnt DESC"

    query = f"MATCH (v:Vendor){match_extra}\nWHERE " + " AND ".join(where_parts)
    if style_clause: query += f"\n  {style_clause}"
    query += f"""
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price,
       v.productPrice AS originalPrice, v.rating AS rating,
       v.reviewCnt AS reviewCnt, v.address AS address,
       v.detailCmt AS description, v.profileUrl AS url
ORDER BY {order}
LIMIT 10"""

    with driver.session() as session:
        rows = [rec.data() for rec in session.run(query, **params)]

    if excludes:
        rows = [r for r in rows if not any(ex.lower() in str(r).lower() for ex in excludes)]
    return rows


def search_with_fallback(driver, parsed):
    region = parsed.get("conditions", {}).get("region")
    if region and not any(r in region for r in AVAILABLE_REGIONS):
        return [], "region_unavailable"

    rows = search_by_conditions(driver, parsed)
    if rows: return rows, "exact"

    conds = parsed.get("conditions", {})
    if conds.get("maxPrice"):
        relaxed = json.loads(json.dumps(parsed))
        relaxed["conditions"]["maxPrice"] = int(conds["maxPrice"] * 1.3)
        rows = search_by_conditions(driver, relaxed)
        if rows: return rows, "price_relaxed"

    if conds.get("styles"):
        relaxed = json.loads(json.dumps(parsed))
        relaxed["conditions"]["styles"] = []
        rows = search_by_conditions(driver, relaxed)
        if rows: return rows, "style_relaxed"

    if conds.get("region"):
        relaxed = json.loads(json.dumps(parsed))
        relaxed["conditions"]["region"] = None
        rows = search_by_conditions(driver, relaxed)
        if rows: return rows, "region_relaxed"

    minimal = {"conditions": {}, "priority": parsed.get("priority", [])}
    rows = search_by_conditions(driver, minimal)
    if rows: return rows, "all_fallback"

    return [], "no_result"


def compare_vendors(driver, names):
    where_parts = ["v.name CONTAINS $name" + str(i) for i in range(len(names))]
    params = {"name" + str(i): n for i, n in enumerate(names)}
    query = "MATCH (v:Vendor) WHERE " + " OR ".join(where_parts) + """
    OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
    RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price,
           v.productPrice AS originalPrice, v.rating AS rating,
           v.reviewCnt AS reviewCnt, v.address AS address,
           v.detailCmt AS description, v.packageInfoStr AS packageInfo,
           collect(DISTINCT t.name) AS tags
    """
    with driver.session() as session:
        return [rec.data() for rec in session.run(query, **params)]


def get_vendor_detail(driver, name):
    query = """
    MATCH (v:Vendor) WHERE v.name CONTAINS $name
    OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
    RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price,
           v.productPrice AS originalPrice, v.rating AS rating,
           v.reviewCnt AS reviewCnt, v.address AS address, v.tel AS tel,
           v.detailCmt AS description, v.packageInfoStr AS packageInfo,
           v.addcostStr AS addcost, v.reviewsStr AS reviews,
           v.holiday AS holiday, collect(DISTINCT t.name) AS tags
    LIMIT 1
    """
    with driver.session() as session:
        rec = session.run(query, name=name).single()
        return rec.data() if rec else None

print("검색 함수 정의 완료")


## 9. 응답 생성 LLM - 드레스

In [ ]:
RESPONSE_SYSTEM_PROMPT = """당신은 웨딩 드레스 전문 추천 챗봇입니다.

[답변 규칙]
1. 검색 결과 데이터 기반으로만 답변. 없는 내용을 만들지 마세요.
2. 추천 시 '왜 이 업체를 추천하는지' 이유를 포함하세요.
3. 가격은 할인가 기준, 정가 대비 할인율도 언급하세요.
4. 드레스 구성(촬영/본식, 벌수, 화이트/컬러)을 구체적으로 안내하세요.
5. 리뷰 수와 평점(100점 만점)을 언급하세요.
6. 답변 마지막에 후속 질문을 유도하세요.
7. 비교 시 표 형태로 정리하세요.
8. 한 번에 5개 이하로 추천하세요.

[Fallback 안내 문구]
- exact: (별도 안내 없이 정상 추천)
- price_relaxed: "정확한 예산에는 맞는 업체가 없어서, 예산을 조금 넓혀 추천드립니다."
- style_relaxed: "해당 조건의 업체가 부족해서, 비슷한 구성도 포함하여 추천드립니다."
- region_unavailable: "현재 해당 지역의 데이터는 보유하고 있지 않습니다. 서울/경기 지역만 제공 가능합니다."
- all_fallback: "세부 조건에 맞는 업체가 없어서, 전체 목록에서 추천드립니다."
"""

def generate_response(message, results, parsed, fallback_type="exact"):
    results_str = json.dumps(results[:5], ensure_ascii=False, indent=2)[:4000]
    prompt = f"사용자 질문: {message}\n\n파싱: {json.dumps(parsed, ensure_ascii=False)}\n\n검색 모드: {fallback_type}\n\n결과:\n{results_str}"
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": RESPONSE_SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
        temperature=0.3, max_tokens=1500)
    return resp.choices[0].message.content

print("응답 생성 함수 정의 완료")


## 10. Gradio 챗봇 - 드레스

In [ ]:
last_results = {"items": [], "parsed": {}}

def response_fn(message, chat_history):
    global last_results
    chat_history = chat_history or []

    parsed = parse_intent(message, chat_history)
    intent = parsed.get("intent", "GENERAL")
    print(f"[DEBUG] intent={intent}, parsed={json.dumps(parsed, ensure_ascii=False)}")

    if parsed.get("refersPrevious") and last_results["items"]:
        answer = generate_response(message, last_results["items"], parsed, "previous_context")
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})
        return "", chat_history

    try:
        if intent == "PREFERENCE_QUERY":
            pref = get_user_preference(conn, COUPLE_ID)
            if pref:
                lines = [f"- {k}: {v}" for k, v in pref.items() if k != "couple_id" and v]
                answer = "현재 저장된 취향 정보입니다.\n" + "\n".join(lines)
            else:
                answer = "저장된 취향 정보가 없습니다."

        elif intent == "LIKE_QUERY":
            likes = get_user_likes(conn, COUPLE_ID)
            if likes:
                lines = [f"- {l.get('name','알수없음')} ({l.get('category','')})" for l in likes]
                answer = f"좋아요한 업체가 {len(likes)}건 있습니다.\n" + "\n".join(lines)
            else:
                answer = "좋아요한 업체가 없습니다."

        elif intent == "LIKE_BASED_RECOMMEND":
            likes = get_user_likes(conn, COUPLE_ID)
            if likes:
                results = compare_vendors(driver, [likes[0].get("name", "")])
                if results:
                    last_results = {"items": results, "parsed": parsed}
                    answer = generate_response(message, results, parsed, "like_based")
                else:
                    answer = "좋아요 기반 추천 결과를 찾지 못했습니다."
            else:
                answer = "좋아요 기록이 없어 추천할 수 없습니다."

        elif intent == "COMPARE":
            names = parsed.get("comparison", [])
            if len(names) >= 2:
                results = compare_vendors(driver, names)
                if len(results) >= 2:
                    answer = generate_response(message, results, parsed, "compare")
                else:
                    found = [r["name"] for r in results] if results else []
                    not_found = [n for n in names if not any(n in f for f in found)]
                    answer = f"다음 업체를 찾지 못했습니다: {', '.join(not_found)}\n정확한 업체명으로 다시 시도해주세요."
            else:
                answer = "비교할 업체 이름을 두 개 이상 알려주세요.\n예: '셀렉션H랑 모네뜨아르 비교해줘'"

        elif intent == "INFO":
            names = parsed.get("comparison", [])
            if names:
                detail = get_vendor_detail(driver, names[0])
                if detail:
                    answer = generate_response(message, [detail], parsed, "detail")
                else:
                    answer = f"'{names[0]}' 업체 정보를 찾지 못했습니다."
            else:
                try:
                    rag_result = rag.search(query_text=message)
                    answer = rag_result.answer
                except:
                    answer = "어떤 업체에 대해 알고 싶으신가요? 업체명을 포함해서 질문해주세요."

        elif intent == "RECOMMEND":
            results, fallback_type = search_with_fallback(driver, parsed)
            if fallback_type == "region_unavailable":
                region = parsed.get("conditions", {}).get("region", "")
                answer = f"현재 {region} 지역 드레스샵 데이터는 보유하고 있지 않습니다.\n서울/경기 지역 약 88개 업체만 제공 가능합니다."
            elif results:
                last_results = {"items": results, "parsed": parsed}
                answer = generate_response(message, results, parsed, fallback_type)
            else:
                answer = "현재 조건에 맞는 업체를 찾지 못했습니다. 조건을 조금 넓혀볼까요?"

        else:
            try:
                rag_result = rag.search(query_text=message)
                answer = rag_result.answer
            except Exception as e:
                print(f"[RAG 실패] {e}")
                resp = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[{"role": "system", "content": "당신은 웨딩 드레스 전문 상담사입니다. 서울/경기 88개 업체 데이터를 보유하고 있습니다."}, {"role": "user", "content": message}],
                    temperature=0.5, max_tokens=1000)
                answer = resp.choices[0].message.content

    except Exception as e:
        print(f"[ERROR] {e}")
        answer = "죄송합니다. 처리 중 오류가 발생했습니다. 다시 시도해주세요."

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})
    return "", chat_history


with gr.Blocks(title="드레스 추천 챗봇") as demo:
    gr.Markdown("# 드레스 추천 챗봇\n웨딩 드레스 추천을 도와드립니다! (서울/경기 88개 업체)")
    chatbot = gr.Chatbot(height=600, type="messages")
    with gr.Row():
        msg = gr.Textbox(placeholder="예: 200만원 이하 촬영+본식 드레스 추천해줘", show_label=False, scale=9)
        send_btn = gr.Button("전송", scale=1)
    gr.Markdown("### 이런 질문을 해보세요")
    with gr.Row():
        gr.Button("드레스 추천해줘").click(lambda: ("드레스 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("200만원 이하").click(lambda: ("200만원 이하 드레스 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("촬영+본식 4벌").click(lambda: ("촬영+본식 드레스 4벌 이상 추천", None), outputs=[msg, chatbot])
        gr.Button("리뷰 좋은 곳").click(lambda: ("리뷰 좋은 드레스샵 추천해줘", None), outputs=[msg, chatbot])
    with gr.Row():
        gr.Button("인기 드레스샵").click(lambda: ("요즘 인기있는 드레스샵 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("저렴한 순").click(lambda: ("가장 저렴한 드레스 5개 알려줘", None), outputs=[msg, chatbot])
        gr.Button("본식 드레스만").click(lambda: ("본식 드레스만 따로 있는 곳 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("셀렉션H 상세").click(lambda: ("셀렉션H 가격이랑 구성 알려줘", None), outputs=[msg, chatbot])
    msg.submit(response_fn, [msg, chatbot], [msg, chatbot])
    send_btn.click(response_fn, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)
